# Models

In [5]:
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

CURRENT_DIR = Path.cwd()

CleanedVehicle=pd.read_csv('../Data/Processed/Cleaned_EV_Vehicle_Specs.csv')
CleanedVehicle.drop('Unnamed: 0', axis=1, inplace=True)
CleanedVehicle.drop('Unnamed: 0.1', axis=1, inplace=True)

In [6]:
CleanedVehicle

,Engine_Cylinders,Engine_Size_L,City_MPG,Highway_MPG,Combined_MPG,CO2_Emissions_g_per_mile,EV_Range_miles,Vehicle_Category,Fuel_Efficiency_Score,Power Proxy,CO2_per_MPG
0,6.0,3.8,16,23,19,472,0,4,19.333333,22.8,24.842105
1,5.0,2.5,20,28,23,378,0,4,23.666667,12.5,16.434783
2,4.0,2.5,26,37,30,298,0,4,31.000000,10.0,9.933333
3,4.0,2.5,28,39,32,280,0,4,33.000000,10.0,8.750000
4,4.0,2.5,25,35,29,308,0,4,29.666667,10.0,10.620690
...,...,...,...,...,...,...,...,...,...,...,...
15295,0.0,0.0,127,107,117,0,287,1,117.000000,0.0,0.000000
15296,0.0,0.0,122,102,112,0,273,1,112.000000,0.0,0.000000
15297,8.0,4.0,14,19,16,486,280,4,16.333333,32.0,30.375000
15298,4.0,2.4,27,27,27,97,420,4,27.000000,9.6,3.592593


## **Model 1: Logistic Regression**

In [75]:
from sklearn.linear_model import LogisticRegression

lr_model=LogisticRegression(solver='saga', max_iter=1000)
X=CleanedVehicle.drop('Vehicle_Category', axis=1)
Y=CleanedVehicle['Vehicle_Category']

#Splitting the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=10)

#Normalizing the data using StandardScaler
ss=StandardScaler()
X_trainSS=ss.fit_transform(X_train)
X_testSS=ss.transform(X_test)

lr_model.fit(X_trainSS, Y_train)
Y_Prediction=lr_model.predict(X_testSS)

In [76]:
joblib.dump(ss, CURRENT_DIR.parent/'Artifacts'/'StandardScaler_Logistic_Regression.joblib')
joblib.dump(lr_model, CURRENT_DIR.parent/'Artifacts'/'Logistic_Regression_Model.joblib')

['C:\\Users\\nelly\\Documents\\Code\\Python\\Personal Projects\\Data Science\\EV Vehicle Classification Project\\Artifacts\\Logistic_Regression_Model.joblib']

In [10]:
X_testSS,Y_test

NameError: name 'X_testSS' is not defined

In [78]:
AccuracyTrain=accuracy_score(Y_train, lr_model.predict(X_trainSS))
print(f'Accuracy of Logistic Regression Model on Training Set: {AccuracyTrain*100:.2f}%')
Accuracy=accuracy_score(Y_test, Y_Prediction)
print(f'Accuracy of Logistic Regression Model: {Accuracy*100:.2f}%')
#The logistic regression model achieved an accuracy of approximately 85.00% on the test set, which indicates that it is able to classify the vehicle categories with a good level of accuracy. However, it is important to also look at other metrics such as precision, recall, and F1-score to get a more comprehensive understanding of the model's performance, especially if there is class imbalance in the dataset.

Accuracy of Logistic Regression Model on Training Set: 99.34%
Accuracy of Logistic Regression Model: 99.28%


In [79]:
print("Classification Report:\n", classification_report(Y_test, Y_Prediction, target_names=['EV', 'Hybrid', 'ICE (Diesel)', 'ICE (Gas)']))
#Model is highly effective in classifying EVs and ICE vehicles, but struggles with hybrids, which may be due to the limited number of hybrid samples in the dataset.

Classification Report:
               precision    recall  f1-score   support

          EV       0.97      1.00      0.98       270
      Hybrid       0.50      0.11      0.18         9
ICE (Diesel)       1.00      0.77      0.87        56
   ICE (Gas)       1.00      1.00      1.00      2725

    accuracy                           0.99      3060
   macro avg       0.87      0.72      0.76      3060
weighted avg       0.99      0.99      0.99      3060



In [80]:
print("Confusion Matrix:\n", confusion_matrix(Y_test, Y_Prediction))
#The confusion matrix shows that the model is able to correctly classify a majority of EVs and ICE vehicles, but there are some misclassifications, particularly with hybrids. This suggests that the model may benefit from additional data or feature engineering to improve its ability to distinguish between hybrids and other vehicle categories.

Confusion Matrix:
 [[ 269    1    0    0]
 [   8    1    0    0]
 [   0    0   43   13]
 [   0    0    0 2725]]


## **Model 2:Decision Tree Classifier**

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

X=CleanedVehicle.drop('Vehicle_Category', axis=1)
Y=CleanedVehicle['Vehicle_Category']

#Splitting the data into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=10)


dt_model=DecisionTreeClassifier(max_depth=3, random_state=10)
dt_model.fit(X_train, Y_train)
Y_Prediction_DT=dt_model.predict(X_test)

In [9]:
print(f"Test Predictions:{Y_Prediction_DT}")


Test Predictions:[4 4 4 ... 1 4 4]


In [12]:
AccuracyTest=accuracy_score(Y_test,Y_Prediction_DT)
AccuracyTest

0.9859477124183007